<a href="https://colab.research.google.com/github/moazamzf/code-switching-codesaviours-si26--Moazam-/blob/main/SI26_Week8_moazam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1 — Install dependencies
!pip install -q transformers datasets seqeval huggingface_hub torch scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# CELL 2 — Imports
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [ ]:
# CELL 3 — Load your published dataset from HuggingFace
dataset = load_dataset("Moazamzf/code-switching-codesaviours-si26-moazam")
print(dataset)
print(dataset["train"][0])

# Your dataset is ONE ROW PER WORD (columns: 'sentence', 'word', 'label'),
# not one row per sentence. We need to group words back into per-sentence
# lists before we can tokenize, so the columns below feed CELL 3b.
SENTENCE_COL = "sentence"
WORD_COL = "word"
LABEL_COL = "label"

# These are the column names the REST of the script uses (after grouping).
# Leave these as-is — CELL 3b below builds them for you.
TOKEN_COL = "tokens"
TAG_COL = "tags"


README.md:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'word', 'label'],
        num_rows: 1296
    })
})
{'sentence': 'Aaj mera mood nahi hai for anything', 'word': 'Aaj', 'label': 'URD'}


In [ ]:
# CELL 3b — Group flat word-level rows into per-sentence tokens/tags lists
import pandas as pd
from datasets import Dataset, DatasetDict

df = dataset["train"].to_pandas()

grouped = (
    df.groupby(SENTENCE_COL, sort=False)
    .agg(tokens=(WORD_COL, list), tags=(LABEL_COL, list))
    .reset_index(drop=True)
)

print(f"Grouped {len(df)} words into {len(grouped)} sentences")
print(grouped.iloc[0].to_dict())  # sanity check: should show a full sentence + its tags

sentence_dataset = Dataset.from_pandas(grouped)
dataset = DatasetDict({"train": sentence_dataset})



Grouped 1296 words into 150 sentences
{'tokens': ['Aaj', 'mera', 'mood', 'nahi', 'hai', 'for', 'anything'], 'tags': ['URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG']}


In [ ]:
# CELL 4 — Build label list and mappings
label_list = sorted(set(tag for ex in dataset["train"][TAG_COL] for tag in ex))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
print("Labels:", label_list)


Labels: ['ENG', 'MIX', 'URD']


In [ ]:

# CELL 5 — Load tokenizer and model
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)



model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:

# CELL 6 — Tokenize + align labels to subwords
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples[TOKEN_COL],
        truncation=True,
        is_split_into_words=True,
    )
    all_labels = []
    for i, tags in enumerate(examples[TAG_COL]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)  # special tokens ([CLS], [SEP], padding)
            elif word_id != prev_word_id:
                label_ids.append(label2id[tags[word_id]])
            else:
                label_ids.append(-100)  # only label the first subword of each word
            prev_word_id = word_id
        all_labels.append(label_ids)
    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [ ]:
# CELL 7 — Train/test split (skip if your dataset already has a test split)
if "test" not in tokenized_dataset:
    split = tokenized_dataset["train"].train_test_split(test_size=0.2, seed=42)
    tokenized_dataset["train"] = split["train"]
    tokenized_dataset["test"] = split["test"]


In [ ]:
# CELL 8 — Metrics function
# NOTE: this is plain per-word classification (not NER chunking), so we flatten
# everything into simple lists and use sklearn — seqeval assumes IOB-style
# tags (e.g. "B-PER") and will mis-parse labels like "URD"/"ENG"/"MIX".

import itertools

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    trainer.compute_metrics = compute_metrics

    # Collect valid predictions and labels into a list of lists first
    # Then flatten them explicitly to ensure compatibility with sklearn.metrics
    predictions_per_sentence = []
    labels_per_sentence = []

    for pred_sentence, label_sentence in zip(predictions, labels):
        current_predictions = []
        current_labels = []
        for p, l in zip(pred_sentence, label_sentence):
            if l != -100:
                current_predictions.append(id2label[p])
                current_labels.append(id2label[l])
        predictions_per_sentence.append(current_predictions)
        labels_per_sentence.append(current_labels)

    # Flatten the lists of lists
    true_predictions = list(itertools.chain.from_iterable(predictions_per_sentence))
    true_labels = list(itertools.chain.from_iterable(labels_per_sentence))

    return {
        "accuracy": accuracy_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions, average="weighted"),
    }

In [ ]:

# CELL 9 — Training arguments
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,  # we'll push manually in CELL 12 after checking results
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [ ]:

# CELL 10 — Train
trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.593670,0.657588,0.105769
2,No log,0.399534,0.926070,0.829721
3,No log,0.296223,0.945525,0.888218
4,No log,0.262608,0.945525,0.888218
5,No log,0.252304,0.945525,0.891566


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: URD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: MIX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: URD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: MIX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: URD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: MIX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: URD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: MIX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=40, training_loss=0.4744071960449219, metrics={'train_runtime': 754.9412, 'train_samples_per_second': 0.795, 'train_steps_per_second': 0.053, 'total_flos': 5119829614944.0, 'train_loss': 0.4744071960449219, 'epoch': 5.0})

In [ ]:
   # CELL 11 — Evaluate and print final report (save these numbers for your README!)
   import itertools # Import itertools for robust flattening

   # Get raw predictions and labels from the trainer
   predictions_output, labels_output, _ = trainer.predict(tokenized_dataset["test"])
   # Apply argmax to get the predicted label IDs
   predictions_argmax = np.argmax(predictions_output, axis=2)

   # Process predictions and labels sentence by sentence, filtering out special tokens (-100)
   sentence_level_predictions = []
   sentence_level_labels = []

   for pred_s, label_s in zip(predictions_argmax, labels_output):
       current_preds_for_sentence = []
       current_labels_for_sentence = []
       for p_token_id, l_token_id in zip(pred_s, label_s):
           if l_token_id != -100:
               current_preds_for_sentence.append(id2label[p_token_id])
               current_labels_for_sentence.append(id2label[l_token_id])
       sentence_level_predictions.append(current_preds_for_sentence)
       sentence_level_labels.append(current_labels_for_sentence)

   # Flatten the lists of lists into single lists for sklearn metrics
   flat_true_predictions = list(itertools.chain.from_iterable(sentence_level_predictions))
   flat_true_labels = list(itertools.chain.from_iterable(sentence_level_labels))

   print("Accuracy:", accuracy_score(flat_true_labels, flat_true_predictions))
   print("F1 Score (weighted):", f1_score(flat_true_labels, flat_true_predictions, average="weighted"))
   print(classification_report(flat_true_labels, flat_true_predictions))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Accuracy: 0.9455252918287937
F1 Score (weighted): 0.92944644547271
              precision    recall  f1-score   support

         ENG       0.88      0.99      0.93        87
         MIX       0.00      0.00      0.00         9
         URD       0.99      0.98      0.98       161

    accuracy                           0.95       257
   macro avg       0.62      0.65      0.64       257
weighted avg       0.92      0.95      0.93       257



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:

# CELL 12 — Login and push model to HuggingFace Hub
from huggingface_hub import notebook_login
notebook_login()  # paste your HF write-access token when prompted

HUB_MODEL_NAME = "code-switching-si26-model"  # change if you want a different repo name

trainer.save_model(f"./{HUB_MODEL_NAME}")
tokenizer.save_pretrained(f"./{HUB_MODEL_NAME}")

model.push_to_hub(HUB_MODEL_NAME)
tokenizer.push_to_hub(HUB_MODEL_NAME)

print(f"Model pushed! Check: https://huggingface.co/Moazamzf/{HUB_MODEL_NAME}")



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...35mb9nj/model.safetensors:   0%|          | 13.6kB / 1.11GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpd8dq3b4q/tokenizer.json:  47%|####6     | 7.96MB / 17.1MB            

Model pushed! Check: https://huggingface.co/Moazamzf/code-switching-si26-model
